[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-semi-supervised.ipynb)

# Semi-Supervised Learning

*AIBits Academy · Machine Learning End To End · ⚠ Advanced Topic*

Every model so far has assumed either every training row has a label (supervised) or none do (unsupervised). Semi-supervised learning lives in the much more common middle ground: a handful of labels, and a lot of unlabeled data too valuable to throw away.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

> **⚠ Why This Page Is Marked "Advanced"**
>
> Semi-supervised learning assumes comfort with both supervised classifiers (Logistic Regression, SVM — Chapters 18 & 26) and the KNN-style distance/graph reasoning from the K-Nearest Neighbours and Hierarchical Clustering chapters. It also builds directly on the idea of the "manifold" first introduced conceptually on the PCA page.

## The Semi-Supervised Scenario

Labels are expensive. A Surat-based textile exporter, **Mehta Textiles**, runs 280 fabric batches through production every month, but only sends a small fraction to the in-house lab for a formal pass/fail quality certificate — lab testing takes time and a technician's attention, so most batches ship on visual inspection alone, uncertified. Throwing away the uncertified 92% of batches and training only on the labeled 8% wastes almost the entire dataset. Semi-supervised learning uses the *structure* of the unlabeled batches — which ones look similar to which — to propagate the few known labels outward.

$$\text{Training set: } \{(x_1,y_1),\dots,(x_\ell,y_\ell)\} \text{ labeled} \ + \ \{x_{\ell+1},\dots,x_n\} \text{ unlabeled, with } \ell \ll n$$

## Label Propagation & Label Spreading

Both algorithms build a similarity graph over *all* n points (labeled and unlabeled together) — typically a k-nearest-neighbours graph — then let label information flow along graph edges, iteratively, until it converges. Points that are graph-close to a labeled "pass" batch become more "pass"-like themselves, even if they were never tested.

$$\begin{gathered}\textbf{Label Propagation} \text{ (hard clamping): } \hat{Y} \leftarrow D^{-1}WY, \text{ then reset labeled rows to their true label every iteration} \\[6pt] \textbf{Label Spreading} \text{ (soft clamping): } \hat{Y} \leftarrow \alpha D^{-1/2}WD^{-1/2}\hat{Y} + (1-\alpha)Y, \text{ labeled rows are nudged, not pinned — more robust when a few labels are noisy}\end{gathered}$$

W is the pairwise similarity (edge weight) matrix and D is the diagonal degree matrix — the same graph-Laplacian machinery that underlies spectral clustering, met on the Advanced Clustering page.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.semi_supervised import LabelPropagation, LabelSpreading
from sklearn.linear_model import LogisticRegression
import numpy as np

# 4 features: yarn tension, dye consistency, weave density, moisture %
X, y = make_classification(n_samples=400, n_features=4, n_informative=3,
                            n_redundant=1, n_clusters_per_class=1, class_sep=1.8, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# Only 8% of batches were ever lab-certified — mask the rest with -1 (sklearn's "unlabeled" sentinel)
rng = np.random.RandomState(7)
n_labeled = int(0.08 * len(y_train))
labeled_idx = rng.choice(len(y_train), size=n_labeled, replace=False)
y_semi = np.full(len(y_train), -1)
y_semi[labeled_idx] = y_train[labeled_idx]

# Baseline: train on ONLY the 8% that were actually lab-certified
baseline = LogisticRegression(max_iter=1000).fit(X_train[labeled_idx], y_train[labeled_idx])
print(f"Labeled-only baseline (8% of batches):  {baseline.score(X_test, y_test):.4f}")

lp = LabelPropagation(kernel='knn', n_neighbors=7, max_iter=5000).fit(X_train, y_semi)
print(f"Label Propagation:                      {lp.score(X_test, y_test):.4f}")

ls = LabelSpreading(kernel='knn', n_neighbors=7, alpha=0.2, max_iter=5000).fit(X_train, y_semi)
print(f"Label Spreading:                         {ls.score(X_test, y_test):.4f}")

With **92% of the true labels thrown away**, Label Propagation still reaches 90.83% test accuracy — matching what a fully-labeled dataset achieves (see the comparison table below) — purely by exploiting which unlabeled batches sit close to which labeled ones in feature space.

## Self-Training — The Practical Cousin of S3VM/TSVM

Semi-supervised SVMs (S3VM/TSVM) search for the decision boundary that maximises margin with respect to *both* labeled and unlabeled points simultaneously — an elegant idea, but the resulting optimisation is non-convex and has no standard scikit-learn implementation. In practice, most teams reach for **self-training** instead: fit a model on the labeled data, use it to predict the unlabeled data, add its most confident predictions to the labeled pool as "pseudo-labels," and repeat. It captures the same iterative spirit — using the model's own evolving decision boundary to progressively label the unlabeled set — with an ordinary classifier underneath.

In [ ]:
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.svm import SVC

base = SVC(probability=True, kernel='rbf', gamma='scale', random_state=42)
# Only pseudo-label points the model is at least 75% confident about, each round
self_train = SelfTrainingClassifier(base, threshold=0.75).fit(X_train, y_semi)
print(f"Self-Training SVM:  {self_train.score(X_test, y_test):.4f}")

| Approach | Labels used | Test accuracy |
|---|---|---|
| Labeled-only baseline | 8% | 0.8000 |
| Label Spreading | 8% + graph structure | 0.8333 |
| Self-Training SVM | 8% + iterative pseudo-labels | 0.8667 |
| Label Propagation | 8% + graph structure | **0.9083** |
| Fully-supervised upper bound | 100% | 0.9083 *(for reference — not achievable here)* |

## Visualizing the Gain

The 92%-unlabeled cost of certifying every batch, versus what each semi-supervised approach recovers from just the 8% Mehta Textiles actually tested:

## What CPLE and S3VM/TSVM Add — Conceptually

Two further approaches, not run live here since they either require original hand-derived implementations (no maintained scikit-learn version exists) or a very different optimisation solver than anything used elsewhere in this course:

- **Generative Gaussian mixtures for SSL** — fit a Gaussian Mixture Model (met on the GMM page) to *all* the data, labeled and unlabeled together, then use the few labeled points only to decide which mixture component corresponds to which class. The unlabeled points shape the component boundaries; the labels just supply the names.
- **Contrastive Pessimistic Likelihood Estimation (CPLE)** — a safety-focused refinement: instead of trusting the unlabeled data's inferred labels at face value, CPLE optimises for the *worst-case* improvement over a supervised-only baseline, guaranteeing performance never degrades below the labeled-only model no matter how the unlabeled data is labeled internally.
- **S3VM / TSVM** — extend the SVM's max-margin objective (Chapter 26) to also penalise the decision boundary for cutting directly through a dense region of *unlabeled* points, on the assumption that a good boundary should pass through low-density regions of the overall data cloud, not just separate the labeled examples.

## The Manifold Assumption

Every graph-based method on this page rests on one assumption: points that are close together on the underlying data manifold are likely to share a label. This is exactly the same assumption that makes t-SNE and UMAP (mentioned on the PCA page) useful for visualisation — both exploit local neighbourhood structure rather than global linear directions. When the assumption fails (e.g., the "similar-looking" batches actually differ on a dimension the feature set doesn't capture — a hidden humidity sensor fault, say), label propagation confidently propagates the *wrong* label just as fast as the right one; there is no built-in safeguard against a misleading similarity graph.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Hide most of the labels

Create `y_semi`: a copy of `y` where a random 90% of the entries are replaced by `-1` (scikit-learn's marker for 'unlabelled'). Keep the 10% chosen by `rng.random(len(y)) < 0.10`. Store the unlabelled fraction in `frac_hidden`.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
X, y = make_moons(n_samples=300, noise=0.1, random_state=0)
rng = np.random.default_rng(0)
y_semi = frac_hidden = None   # TODO


In [ ]:
try:
    check("about 90% hidden", 0.85 < frac_hidden < 0.95)
    check("labels kept where not hidden", ((y_semi == y) | (y_semi == -1)).all())
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.datasets import make_moons
X, y = make_moons(n_samples=300, noise=0.1, random_state=0)
rng = np.random.default_rng(0)
keep = rng.random(len(y)) < 0.10
y_semi = np.where(keep, y, -1)
frac_hidden = float((y_semi == -1).mean())

```

</details>

### Exercise 2 · Medium · Label spreading

Fit `LabelSpreading(kernel="knn", n_neighbors=7)` on `X, y_semi` and store the accuracy of its transduced labels (`.transduction_`) against the true `y` in `acc_ls`.

In [ ]:
from sklearn.semi_supervised import LabelSpreading
acc_ls = None   # TODO


In [ ]:
try:
    check("labels spread well", acc_ls > 0.9)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.semi_supervised import LabelSpreading
ls = LabelSpreading(kernel="knn", n_neighbors=7).fit(X, y_semi)
acc_ls = float((ls.transduction_ == y).mean())

```

</details>

### Exercise 3 · Stretch · Does the unlabelled data help?

Train a plain `LogisticRegression` on **only the labelled points** and measure its accuracy on **all** points (`acc_sup`). Set `ssl_wins` = whether label spreading beat it.

In [ ]:
from sklearn.linear_model import LogisticRegression
acc_sup = ssl_wins = None   # TODO


In [ ]:
try:
    check("supervised-only accuracy computed", 0 < acc_sup < 1)
    check("semi-supervised wins on interleaved moons", ssl_wins is True)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.linear_model import LogisticRegression
lab = y_semi != -1
acc_sup = LogisticRegression().fit(X[lab], y[lab]).score(X, y)
ssl_wins = bool(acc_ls > acc_sup)

```

Unlabelled points reveal the *shape* of each class; a linear model trained on a few labels cannot see that structure.

</details>

---
*Back to the course: **Machine Learning End To End → Semi-Supervised Learning**.*